In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
import json


In [12]:
# Generate a synthetic dataset with 50 input features
# Let's create 1000 samples with 50 features each and binary labels
np.random.seed(42)  # For reproducibility

X = np.random.rand(1000, 50)  # 1000 samples, 50 features
y = np.random.randint(2, size=(1000, 1))  # Binary labels (0 or 1)

print(X.shape, y.shape)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# print(y_train.shape, y_test.shape)
# print(X_train.shape, X_test.shape)


(1000, 50) (1000, 1)


In [13]:
# Define a single-layer perceptron model
model = Sequential([
    Dense(1, input_dim=50, activation='relu'),  # One neuron, 50 inputs, sigmoid activation
    # BatchNormalization()
])

# Compile the model
model.compile(optimizer='sgd',  # Stochastic Gradient Descent
              loss='binary_crossentropy',  # Loss function for binary classification
              metrics=['accuracy'])


# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=10, verbose=1)


Epoch 1/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 293us/step - accuracy: 0.4593 - loss: 7.8360  
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 402us/step - accuracy: 0.4799 - loss: 8.3827
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 319us/step - accuracy: 0.4992 - loss: 8.0712
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 332us/step - accuracy: 0.4988 - loss: 8.0781
Epoch 5/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 279us/step - accuracy: 0.4764 - loss: 8.4398
Epoch 6/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 282us/step - accuracy: 0.4860 - loss: 8.2849
Epoch 7/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 280us/step - accuracy: 0.5161 - loss: 7.7998
Epoch 8/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 280us/step - accuracy: 0.4915 - loss: 8.1960
Epoch 9/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 277us/step - accuracy: 0.4657 - loss: 8.6121
Epoch 10/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 362us/step - accuracy: 0.4674 - loss: 8.5843
Epoch 11/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 270us/step - accuracy: 0.5009 - loss: 8.0450
Epoch 12/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 269us/st

In [14]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5340 - loss: 7.5109 
Test Loss: 7.8979
Test Accuracy: 0.5100


In [15]:
# Access the weights
weights, bias = model.get_weights()

# Save the weights to a JSON file
weights_dict = {
    "weights": weights.tolist(),
    "bias": bias.tolist()
}

# print(weights)

In [16]:
with open("perceptron_weights.json", "w") as fw:
    json.dump(weights_dict, fw)

print("Weights and bias saved to perceptron_weights.json")

Weights and bias saved to perceptron_weights.json


In [17]:
# Read the JSON file
with open('perceptron_weights.json', 'r') as fr:
    data = json.load(fr)

# Convert the JSON data to a format suitable for Verilog
with open('weights_values.mem', 'w') as fmem:
    for weight_value in data['weights']:
        weight_valueQ15 = weight_value[0] * 2**15 # Convert to Q15.0 fixed-point format
        weight_valueQ15 = int(weight_valueQ15) # Convert to integer
        # Get the raw binary representation
        binary_representation = bin(weight_valueQ15 & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")
        # print(weight_valueQ15)    
        
    bias_value = data['bias']
    bias_valueQ15 = bias_value[0] * 2**15 # Convert to Q15.0 fixed-point format
    bias_valueQ15 = int(bias_valueQ15) # Convert to integer
    # Get the raw binary representation
    binary_representation = bin(bias_valueQ15 & 0xFFFF)[2:].zfill(16)
    fmem.write(f"{binary_representation}\n") 
    

# Save iput data to a file
XQ15 = X * 2**15
XQ15 = XQ15.astype(np.int16)

with open('input_values.mem', 'w') as fmem:
    for value in XQ15[0]:
        binary_representation = bin(value & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")  # Convert value to float before formatting as binary
# print(XQ15)

In [18]:
# Example single input (make sure it has the correct shape)
single_input = X[0].reshape(1, -1)

# Print the input value
# print("Input value for the single input:", single_input)

# Make a prediction
prediction = model.predict(single_input)

predictionQ15 = prediction[0][0] * 2**15 # Convert to Q15.0 fixed-point format

# Print the prediction and the classified class
print("Prediction for the single input:", prediction)
print("Prediction for the single input in Q15.0 format:", int(predictionQ15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Prediction for the single input: [[0.]]
Prediction for the single input in Q15.0 format: 0


In [19]:
# Assuming single_input and weights are already defined as numpy arrays
# Slice the first ten elements
single_input_ = single_input[0]
weights_ = weights[0:50]

print(single_input_)
print(weights_)

# Perform the dot product with the first ten elements
product = np.dot(single_input_, weights_)
print("Product:", product)
print(product +bias_value[0])

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452
 0.05808361 0.86617615 0.60111501 0.70807258 0.02058449 0.96990985
 0.83244264 0.21233911 0.18182497 0.18340451 0.30424224 0.52475643
 0.43194502 0.29122914 0.61185289 0.13949386 0.29214465 0.36636184
 0.45606998 0.78517596 0.19967378 0.51423444 0.59241457 0.04645041
 0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735
 0.30461377 0.09767211 0.68423303 0.44015249 0.12203823 0.49517691
 0.03438852 0.9093204  0.25877998 0.66252228 0.31171108 0.52006802
 0.54671028 0.18485446]
[[-0.18023333]
 [ 0.20717737]
 [-0.2481536 ]
 [-0.33391505]
 [ 0.28623444]
 [ 0.16895804]
 [ 0.22848435]
 [ 0.09977849]
 [-0.24532023]
 [ 0.05179223]
 [-0.15226525]
 [-0.30821684]
 [-0.02894367]
 [ 0.02183766]
 [ 0.22521003]
 [-0.1626057 ]
 [-0.12243869]
 [-0.34877515]
 [-0.19411337]
 [-0.125154  ]
 [ 0.12493502]
 [ 0.09312896]
 [-0.09150762]
 [ 0.04751484]
 [-0.00378452]
 [-0.00690507]
 [ 0.15557013]
 [-0.17152518]
 [-0.23483565]
 [-0.